In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_flim.tools_phasor as flim
import brighteyes_ism.analysis.APR_lib as apr
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.analysis.Tools_lib as tools
import brighteyes_ism.dataio.mcs as mcs
from scipy.ndimage import shift
from skimage.registration import phase_cross_correlation
from tqdm import tqdm


In [ ]:
h = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\G3BP1_29_05_2024","r")
print(h.keys())
img = h["dataset_1"]
print(img.shape)

In [ ]:
bins = np.sum(img, axis=(0,1))
plt.figure()
for i in range(bins.shape[-1]):
    plt.plot(bins[:, i])

In [ ]:
img_sum = np.sum(img, axis=-2)
img_int = np.asarray(img_sum)
img_int.shape

In [ ]:
pxs = 146         # pixel size of the ISM dataset converted in nm (metadata saves the pixel size in micrometers)
upsampling_factor = 10        # upsampling factor to improve accuracy in simulation
Nch = img_int.shape[-1]
reference = Nch // 2  #the reference image for the shift vectors extraction is the central one

shift_vectors, _ = apr.ShiftVectors(img_int, upsampling_factor, reference, filter_sigma=1)
shift_vectors *= pxs # we convert the shifts from pixels units to nm

In [ ]:
fig_3, ax_3 = plt.subplots(figsize = (10, 10))

fing = tools.fingerprint(img_int)
_ = gr.PlotShiftVectors(shift_vectors, color=fing, fig = fig_3, ax = ax_3)

In [ ]:
import brighteyes_ism.simulation.PSF_sim as sim  
ups = 10
exPar = sim.simSettings()
exPar.na = 1.4   # numerical aperture
exPar.wl = 488   # wavelength [nm]
exPar.gamma = 45 # parameter describing the light polarization
exPar.beta = 90  # parameter describing the light polarization
exPar.n = 1.5    # refractive index of immersion medium
exPar.mask_sampl = 50       #sampling points of the phase mask in simulation

emPar = exPar.copy()
emPar.wl = 510   # wavelength [nm]

grid = sim.GridParameters()
grid.N = 5  # number of elements for each axis of the detector array, assumed squared
grid.pxpitch = 75e3  # detector pixel pitch [nm]
grid.pxdim = 50e3  # detector pixel size [nm]
grid.Nz = 2
grid.pxsizex = pxs  # pixel size of acquisition [nm]
grid.pxsizez = 551  # 'ToFind'


In [ ]:
from s2ism import psf_estimator as est                            
from s2ism import s2ism as s2


In [ ]:
        

#grid_simul.Nx = 100*ups
#psf, det_psf, ex_psf = est.psf_estimator_from_data(img_int, exPar, emPar, grid, downsample = False, z_out_of_focus = 551)
psf, det_psf, ex_psf = est.psf_estimator_from_data(img_int, exPar, emPar, grid)

In [ ]:
gr.ShowDataset(psf[0])    #[0,450:550,450:550]

In [ ]:
#shift_vectors_psf, _ = apr.ShiftVectors(psf[1], upsampling_factor, reference, filter_sigma=1)
#shift_vectors_psf *= pxs # we convert the shifts from pixels units to nm
#fig_3, ax_3 = plt.subplots(figsize = (10, 10))


#_ = gr.PlotShiftVectors(shift_vectors_psf, fig = fig_3, ax = ax_3)

In [ ]:
#psf_down=tools.DownSample(psf, ds = ups, order = 'zxyc')

In [ ]:
#gr.ShowDataset(psf_down[1,45:55,45:55])

In [ ]:
def rect(t, w):
    r = np.where(abs(t) <= w/2, 1, 0)*1.
    r /= r.sum()
    return r

In [ ]:
from skimage.filters import gaussian
def smooth_irf(t, s, w):
    irf_1 = rect(t, w)
    # g = gauss(t, 1, 0, t[len(t)//2], s)
    # irf_2 = convolve(irf_1, g, mode='same')
    dt = t[1] - t[0]
    irf_2 = gaussian(irf_1, s/dt)
    irf_2 /= irf_2.sum()
    return irf_2
 
 
# %%
dt = 48 * 1e-3  # ns
nbin = 260
window = 0.2  # ns
 
time = np.arange(nbin)*dt
 
t0 = time[len(time)//2]
# t0 = nbin*dt + 0
 

 
irf = smooth_irf(time - t0, dt, window)

In [ ]:
plt.figure()
plt.plot(irf)

In [ ]:
print(emPar.airy_unit)

In [ ]:
print(irf.shape)
print(psf.shape)

In [ ]:
irf_1 = np.repeat(irf, 25).reshape((260,25))
print(irf_1.shape)

In [ ]:
psf_tot = est.combine_psf_irf(psf, irf_1)

In [ ]:
print(psf_tot.shape)

In [ ]:
#reconstruction = s2.max_likelihood_reconstruction(img[250:350, 25:125, :, :], psf_tot[:, 1:, 1:], stop='fixed', max_iter = 10)[0]   #[250:350, 25:125, :, :]
img_arr = np.asarray(img)
reconstruction = s2.batch_reconstruction(img_arr, psf_tot, (100,100), 20, stop='fixed',
                                         max_iter=10)

In [ ]:
print(reconstruction[0].shape)

In [ ]:
reconstruction_t_sum = np.sum(reconstruction, axis=(-1))
gr.ShowImg(reconstruction_t_sum[0], pxs)

In [ ]:
gr.ShowImg(reconstruction_t_sum[1], pxs)

In [ ]:
img_s = np.sum(img, axis=(-2, -1))
print(img_s)


In [ ]:
gr.ShowImg(img_s[250:350, 25:125], pxs)

### Lifetime analysis

In [ ]:
data_intensity = np.sum(reconstruction[0], axis=(-1)) #data intensity out of focus
print(data_intensity.shape)

In [ ]:
phasors_pix = flim.phasor(reconstruction[1]) 

In [ ]:
data_intensity_in_focus = np.sum(reconstruction[1], axis=(-1))

### Raw phasor plot of in focus reconstruction

In [ ]:
%matplotlib widget
flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

In [ ]:
phasors_pix_out_of_focus = flim.phasor(reconstruction[0]) 

### Raw phasor plot of out of focus reconstruction

In [ ]:
%matplotlib widget
flim.plot_phasor(phasors_pix_out_of_focus[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### Intensity-based thresholded Phasor plot of in focus reconstruction 

In [ ]:
phasors_pix_thresholded = flim.threshold_phasor(data_intensity_in_focus, phasors_pix, threshold = 0.15)
print(phasors_pix_thresholded.shape)
flim.plot_phasor(phasors_pix_thresholded[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### Intensity-based thresholded Phasor plot of out focus reconstruction 

In [ ]:
phasors_pix_thresholded_out_of_focus = flim.threshold_phasor(data_intensity, phasors_pix_out_of_focus, threshold = 0.15)
print(phasors_pix_thresholded_out_of_focus.shape)
flim.plot_phasor(phasors_pix_thresholded_out_of_focus[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### In focus phasor plot after applying median filter on pixels' phasor (window 3x3)

In [ ]:
phasors_pix_median = flim.median_phasors(phasors_pix, window=3)
flim.plot_phasor(phasors_pix_median[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### Out of focus phasor plot after applying median filter on pixels' phasor (window 3x3)

In [ ]:
phasors_pix_median = flim.median_phasors(phasors_pix_out_of_focus, window=3)
flim.plot_phasor(phasors_pix_median[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

In [ ]:
tau_m_out = flim.calculate_tau_m(np.real(phasors_pix_thresholded_out_of_focus), np.imag(phasors_pix_thresholded_out_of_focus), dfd_freq = 80e6)
print(tau_m_out.shape)

### Lifetime histogram of out of focus image

In [ ]:
tau_m_data = 1e9*tau_m_out.flatten()

plt.figure()
plt.hist(tau_m_data, range = (0, 13), bins = 50)

### Lifetime histogram of in focus image

In [ ]:
tau_m_in = flim.calculate_tau_m(np.real(phasors_pix_thresholded), np.imag(phasors_pix_thresholded), dfd_freq = 80e6)
print(tau_m_in.shape)

In [ ]:
tau_m_data_in = 1e9*tau_m_in.flatten()

plt.figure()
plt.hist(tau_m_data_in, range = (0, 13), bins = 50)